### **De imágenes a videos: representación espacio-temporal y modelos de video**

#### **Del procesamiento espacial al razonamiento temporal reproducible**

Una imagen describe una configuración espacial observada en un instante. Un video describe una función espacio-temporal muestreada, donde la identidad de los objetos, el movimiento, el orden, la duración y la causalidad dependen de varios instantes.

La pregunta central de este cuaderno es:

> **¿Qué cambios matemáticos, arquitectónicos y experimentales son necesarios para transformar un modelo de imágenes en un modelo capaz de representar videos?**

El objetivo no es presentar un catálogo de arquitecturas. El objetivo es construir criterios para decidir:

1. qué información temporal debe conservarse,
2. cómo se tokeniza un video,
3. qué sesgos inductivos se introducen,
4. cuánto cuesta la atención espacio-temporal,
5. qué fallos aparecen por muestreo, pooling y perturbación del orden,
6. cómo se valida que un modelo utiliza tiempo y no solo apariencia.


### **Preguntas de investigación**

#### **Hipótesis de trabajo**

**H1.** Un encoder de imágenes aplicado frame por frame no constituye por sí mismo un modelo de video.

**H2.** El pooling promedio sobre frames es invariante ante permutaciones y no puede representar orden.

**H3.** Inflar un kernel 2D a 3D preserva la respuesta sobre videos estáticos cuando el kernel temporal se normaliza.

**H4.** La tokenización mediante tubelets reduce longitud de secuencia, pero puede eliminar eventos más breves que la extensión temporal del tubelet.

**H5.** La atención espacio-temporal completa ofrece conectividad global, pero su costo cuadrático crece rápidamente con frames y parches.

**H6.** Representaciones con posición temporal, diferencias entre frames o atención condicionada por consulta detectan propiedades que el pooling global pierde.

**H7.** Una evaluación de video es metodológicamente débil si no incluye inversión, barajado, eliminación de frames y cambios en la tasa de muestreo.


### **Configuración reproducible**

#### **Importaciones, rutas y semilla**


In [ ]:
from __future__ import annotations

import csv
import json
import platform
import random
from collections.abc import Iterable
from dataclasses import asdict, dataclass
from math import ceil
from pathlib import Path
from typing import Any, Mapping

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# Determina una ruta estable para guardar los resultados del cuaderno
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "Semana12":
    WEEK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "Semana12").exists():
    WEEK_DIR = CURRENT_DIR / "Semana12"
else:
    WEEK_DIR = CURRENT_DIR

print("Directorio de trabajo:", WEEK_DIR)

@dataclass(frozen=True)
class ExperimentMetadata:
    """Describe la configuración mínima de un experimento reproducible."""

    course: str
    week: str
    notebook: str
    topic: str
    seed: int
    mode: str = "CPU sin servicios externos"
    hardware: str = platform.platform()

def set_seed(seed: int) -> int:
    """Fija semillas para Python y NumPy."""
    random.seed(seed)
    np.random.seed(seed)
    return seed

def ensure_directory(path: str | Path) -> Path:
    """Crea un directorio y devuelve su ruta normalizada."""
    directory = Path(path)
    directory.mkdir(parents=True, exist_ok=True)
    return directory

def save_json(data: Any, path: str | Path) -> Path:
    """Guarda datos en JSON con codificación UTF-8."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
    return output_path

def save_jsonl(records: Iterable[Mapping[str, Any]], path: str | Path) -> Path:
    """Guarda registros en formato JSON Lines."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(dict(record), ensure_ascii=False) + "\n")
    return output_path

def save_csv(rows: Iterable[Mapping[str, Any]], path: str | Path) -> Path:
    """Guarda una colección de diccionarios en CSV."""
    row_list = [dict(row) for row in rows]
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not row_list:
        output_path.write_text("", encoding="utf-8")
        return output_path
    fieldnames = list(row_list[0].keys())
    with output_path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(row_list)
    return output_path

def metadata_to_dict(metadata: ExperimentMetadata) -> dict[str, Any]:
    """Convierte metadatos a un diccionario serializable."""
    return asdict(metadata)

def _validate_interval(interval: tuple[float, float]) -> tuple[float, float]:
    """Valida un intervalo temporal cerrado por la izquierda."""
    start, end = float(interval[0]), float(interval[1])
    if end <= start:
        raise ValueError("El final debe ser mayor que el inicio")
    return start, end

def temporal_iou(
    predicted_interval: tuple[float, float],
    reference_interval: tuple[float, float],
) -> float:
    """Calcula intersección sobre unión temporal."""
    predicted_start, predicted_end = _validate_interval(predicted_interval)
    reference_start, reference_end = _validate_interval(reference_interval)
    intersection = max(
        0.0,
        min(predicted_end, reference_end) - max(predicted_start, reference_start),
    )
    union = (
        predicted_end - predicted_start
        + reference_end - reference_start
        - intersection
    )
    return float(intersection / union) if union > 0 else 0.0

def boundary_error(
    predicted_interval: tuple[float, float],
    reference_interval: tuple[float, float],
) -> tuple[float, float]:
    """Calcula errores absolutos de inicio y final."""
    predicted_start, predicted_end = _validate_interval(predicted_interval)
    reference_start, reference_end = _validate_interval(reference_interval)
    return abs(predicted_start - reference_start), abs(predicted_end - reference_end)

def center_error(
    predicted_interval: tuple[float, float],
    reference_interval: tuple[float, float],
) -> float:
    """Calcula el error absoluto entre centros temporales."""
    predicted_start, predicted_end = _validate_interval(predicted_interval)
    reference_start, reference_end = _validate_interval(reference_interval)
    predicted_center = (predicted_start + predicted_end) / 2.0
    reference_center = (reference_start + reference_end) / 2.0
    return abs(predicted_center - reference_center)

def duration_error(
    predicted_interval: tuple[float, float],
    reference_interval: tuple[float, float],
) -> float:
    """Calcula el error absoluto de duración."""
    predicted_start, predicted_end = _validate_interval(predicted_interval)
    reference_start, reference_end = _validate_interval(reference_interval)
    return abs((predicted_end - predicted_start) - (reference_end - reference_start))

def offset_mean_absolute_error(
    predicted_offsets: Iterable[float],
    reference_offsets: Iterable[float],
) -> float:
    """Calcula el error absoluto medio de los desfases."""
    predicted = np.asarray(list(predicted_offsets), dtype=np.float64)
    reference = np.asarray(list(reference_offsets), dtype=np.float64)
    if predicted.shape != reference.shape or predicted.size == 0:
        raise ValueError("Las colecciones de desfases deben tener igual tamaño")
    return float(np.abs(predicted - reference).mean())

def accuracy_with_tolerance(
    predicted_offsets: Iterable[float],
    reference_offsets: Iterable[float],
    tolerance: float,
) -> float:
    """Calcula exactitud dentro de una tolerancia temporal."""
    if tolerance < 0:
        raise ValueError("La tolerancia no puede ser negativa")
    predicted = np.asarray(list(predicted_offsets), dtype=np.float64)
    reference = np.asarray(list(reference_offsets), dtype=np.float64)
    if predicted.shape != reference.shape or predicted.size == 0:
        raise ValueError("Las colecciones de desfases deben tener igual tamaño")
    return float((np.abs(predicted - reference) <= tolerance).mean())

def bootstrap_mean_interval(
    values: Iterable[float],
    confidence: float = 0.95,
    iterations: int = 2000,
    seed: int = 211,
) -> tuple[float, float, float]:
    """Estima media e intervalo bootstrap percentil."""
    data = np.asarray(list(values), dtype=np.float64)
    if data.size == 0:
        raise ValueError("Se requiere al menos un valor")
    if not 0.0 < confidence < 1.0:
        raise ValueError("La confianza debe estar entre cero y uno")
    if iterations <= 0:
        raise ValueError("Las iteraciones deben ser positivas")
    rng = np.random.default_rng(seed)
    means = np.empty(iterations, dtype=np.float64)
    for index in range(iterations):
        sample = rng.choice(data, size=len(data), replace=True)
        means[index] = sample.mean()
    alpha = (1.0 - confidence) / 2.0
    lower, upper = np.quantile(means, [alpha, 1.0 - alpha])
    return float(data.mean()), float(lower), float(upper)

def _validate_sequence(sequence: np.ndarray) -> np.ndarray:
    """Valida que la secuencia tenga dos dimensiones."""
    array = np.asarray(sequence, dtype=np.float64)
    if array.ndim != 2 or array.shape[0] == 0:
        raise ValueError("La secuencia debe tener forma tiempo por dimensión")
    return array

def mean_pool(sequence: np.ndarray) -> np.ndarray:
    """Calcula el promedio temporal de una secuencia."""
    array = _validate_sequence(sequence)
    return array.mean(axis=0)

def max_pool(sequence: np.ndarray) -> np.ndarray:
    """Calcula el máximo temporal por dimensión."""
    array = _validate_sequence(sequence)
    return array.max(axis=0)

def weighted_pool(sequence: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Combina segmentos mediante pesos temporales normalizados."""
    array = _validate_sequence(sequence)
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim != 1 or len(weight_array) != len(array):
        raise ValueError("Los pesos deben tener una entrada por instante")
    total = weight_array.sum()
    if total <= 0:
        raise ValueError("La suma de pesos debe ser positiva")
    normalized = weight_array / total
    return normalized @ array

def position_aware_pool(sequence: np.ndarray) -> np.ndarray:
    """Combina contenido y posición para conservar sensibilidad al orden."""
    array = _validate_sequence(sequence)
    positions = np.linspace(-1.0, 1.0, len(array))
    content = array.mean(axis=0)
    temporal_moment = positions @ array / len(array)
    return np.concatenate([content, temporal_moment])

def attention_pool(sequence: np.ndarray, query: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Aplica atención simple condicionada por una consulta."""
    array = _validate_sequence(sequence)
    query_array = np.asarray(query, dtype=np.float64)
    if query_array.ndim != 1 or query_array.shape[0] != array.shape[1]:
        raise ValueError("La consulta no coincide con la dimensión de la secuencia")
    scores = array @ query_array
    scores = scores - scores.max()
    weights = np.exp(scores)
    weights = weights / max(weights.sum(), 1e-12)
    return weights @ array, weights

@dataclass(frozen=True)
class TokenizationSummary:
    """Resume la tokenización espacial o espacio-temporal."""

    token_count: int
    token_dimension: int
    temporal_tokens: int
    spatial_tokens: int

def _validate_image(image: np.ndarray) -> np.ndarray:
    """Valida una imagen con forma alto por ancho por canales."""
    array = np.asarray(image, dtype=np.float64)
    if array.ndim != 3:
        raise ValueError("La imagen debe tener forma alto por ancho por canales")
    if min(array.shape) <= 0:
        raise ValueError("La imagen no puede contener dimensiones vacías")
    return array

def _validate_video(video: np.ndarray) -> np.ndarray:
    """Valida un video con forma tiempo por alto por ancho por canales."""
    array = np.asarray(video, dtype=np.float64)
    if array.ndim != 4:
        raise ValueError("El video debe tener forma tiempo por alto por ancho por canales")
    if min(array.shape) <= 0:
        raise ValueError("El video no puede contener dimensiones vacías")
    return array

def patchify_image(image: np.ndarray, patch_size: int) -> np.ndarray:
    """Convierte una imagen en una secuencia de parches aplanados."""
    array = _validate_image(image)
    if patch_size <= 0:
        raise ValueError("El tamaño del parche debe ser positivo")

    height, width, channels = array.shape
    if height % patch_size != 0 or width % patch_size != 0:
        raise ValueError("El alto y el ancho deben ser divisibles por el tamaño del parche")

    patches = array.reshape(
        height // patch_size,
        patch_size,
        width // patch_size,
        patch_size,
        channels,
    )
    patches = patches.transpose(0, 2, 1, 3, 4)
    return patches.reshape(-1, patch_size * patch_size * channels)

def tubeletify_video(
    video: np.ndarray,
    tubelet_size: int,
    patch_size: int,
) -> np.ndarray:
    """Convierte un video en tubelets espacio-temporales aplanados."""
    array = _validate_video(video)
    if tubelet_size <= 0 or patch_size <= 0:
        raise ValueError("Los tamaños temporal y espacial deben ser positivos")

    frames, height, width, channels = array.shape
    if frames % tubelet_size != 0:
        raise ValueError("La cantidad de frames debe ser divisible por el tamaño temporal")
    if height % patch_size != 0 or width % patch_size != 0:
        raise ValueError("El alto y el ancho deben ser divisibles por el tamaño del parche")

    tubelets = array.reshape(
        frames // tubelet_size,
        tubelet_size,
        height // patch_size,
        patch_size,
        width // patch_size,
        patch_size,
        channels,
    )
    tubelets = tubelets.transpose(0, 2, 4, 1, 3, 5, 6)
    return tubelets.reshape(
        -1,
        tubelet_size * patch_size * patch_size * channels,
    )

def summarize_image_tokens(
    image_shape: tuple[int, int, int],
    patch_size: int,
) -> TokenizationSummary:
    """Calcula el número y la dimensión de tokens de una imagen."""
    height, width, channels = image_shape
    spatial_tokens = (height // patch_size) * (width // patch_size)
    return TokenizationSummary(
        token_count=spatial_tokens,
        token_dimension=patch_size * patch_size * channels,
        temporal_tokens=1,
        spatial_tokens=spatial_tokens,
    )

def summarize_video_tokens(
    video_shape: tuple[int, int, int, int],
    tubelet_size: int,
    patch_size: int,
) -> TokenizationSummary:
    """Calcula el número y la dimensión de tokens de un video."""
    frames, height, width, channels = video_shape
    temporal_tokens = frames // tubelet_size
    spatial_tokens = (height // patch_size) * (width // patch_size)
    return TokenizationSummary(
        token_count=temporal_tokens * spatial_tokens,
        token_dimension=tubelet_size * patch_size * patch_size * channels,
        temporal_tokens=temporal_tokens,
        spatial_tokens=spatial_tokens,
    )

def inflate_2d_kernel(
    kernel_2d: np.ndarray,
    temporal_size: int,
    preserve_static_response: bool = True,
) -> np.ndarray:
    """Expande un kernel bidimensional a una dimensión temporal."""
    kernel = np.asarray(kernel_2d, dtype=np.float64)
    if kernel.ndim != 2:
        raise ValueError("El kernel bidimensional debe tener dos dimensiones")
    if temporal_size <= 0:
        raise ValueError("El tamaño temporal debe ser positivo")

    kernel_3d = np.repeat(kernel[None, :, :], temporal_size, axis=0)
    if preserve_static_response:
        kernel_3d = kernel_3d / temporal_size
    return kernel_3d

def convolve2d_valid(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Aplica una convolución bidimensional válida sobre una imagen en escala de grises."""
    image_array = np.asarray(image, dtype=np.float64)
    kernel_array = np.asarray(kernel, dtype=np.float64)
    if image_array.ndim != 2 or kernel_array.ndim != 2:
        raise ValueError("La imagen y el kernel deben tener dos dimensiones")
    if any(k > s for k, s in zip(kernel_array.shape, image_array.shape)):
        raise ValueError("El kernel no puede ser mayor que la imagen")

    windows = sliding_window_view(image_array, kernel_array.shape)
    return np.tensordot(windows, kernel_array, axes=((2, 3), (0, 1)))

def convolve3d_valid(video: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Aplica una convolución tridimensional válida sobre un video en escala de grises."""
    video_array = np.asarray(video, dtype=np.float64)
    kernel_array = np.asarray(kernel, dtype=np.float64)
    if video_array.ndim != 3 or kernel_array.ndim != 3:
        raise ValueError("El video y el kernel deben tener tres dimensiones")
    if any(k > s for k, s in zip(kernel_array.shape, video_array.shape)):
        raise ValueError("El kernel no puede ser mayor que el video")

    windows = sliding_window_view(video_array, kernel_array.shape)
    return np.tensordot(
        windows,
        kernel_array,
        axes=((3, 4, 5), (0, 1, 2)),
    )

def generate_moving_square_video(
    num_frames: int = 16,
    height: int = 64,
    width: int = 64,
    square_size: int = 12,
    direction: str = "derecha",
    noise_level: float = 0.0,
    seed: int | None = None,
) -> np.ndarray:
    """Genera un video sintético con un cuadrado que se desplaza horizontalmente."""
    if num_frames < 2:
        raise ValueError("El video debe contener al menos dos frames")
    if square_size <= 0 or square_size > min(height, width):
        raise ValueError("El tamaño del cuadrado no es válido")
    if direction not in {"derecha", "izquierda"}:
        raise ValueError("La dirección debe ser derecha o izquierda")
    if noise_level < 0:
        raise ValueError("El nivel de ruido no puede ser negativo")

    rng = np.random.default_rng(seed)
    video = rng.normal(
        0.0,
        noise_level,
        size=(num_frames, height, width, 3),
    )
    video = np.clip(video, 0.0, 1.0)

    y_start = (height - square_size) // 2
    x_positions = np.linspace(0, width - square_size, num_frames)
    if direction == "izquierda":
        x_positions = x_positions[::-1]

    for frame_index, x_position in enumerate(x_positions):
        x_start = int(round(x_position))
        video[
            frame_index,
            y_start : y_start + square_size,
            x_start : x_start + square_size,
            :,
        ] = np.array([1.0, 0.65, 0.15], dtype=np.float64)

    return np.clip(video, 0.0, 1.0)

def frame_difference(video: np.ndarray) -> np.ndarray:
    """Calcula diferencias entre frames consecutivos."""
    array = _validate_video(video)
    return np.diff(array, axis=0)

def extract_frame_features(video: np.ndarray) -> np.ndarray:
    """Extrae intensidad, centroide y dispersión espacial por frame."""
    array = _validate_video(video)
    grayscale = array.mean(axis=-1)
    frames, height, width = grayscale.shape

    x_coordinates = np.linspace(0.0, 1.0, width)
    y_coordinates = np.linspace(0.0, 1.0, height)
    features = []

    for frame in grayscale:
        mass = float(frame.sum())
        if mass <= 1e-12:
            features.append([0.0, 0.5, 0.5, 0.0, 0.0])
            continue

        x_mass = frame.sum(axis=0)
        y_mass = frame.sum(axis=1)
        x_center = float(x_mass @ x_coordinates / mass)
        y_center = float(y_mass @ y_coordinates / mass)
        x_variance = float(x_mass @ ((x_coordinates - x_center) ** 2) / mass)
        y_variance = float(y_mass @ ((y_coordinates - y_center) ** 2) / mass)
        features.append(
            [
                float(frame.mean()),
                x_center,
                y_center,
                x_variance,
                y_variance,
            ]
        )

    result = np.asarray(features, dtype=np.float64)
    if result.shape != (frames, 5):
        raise RuntimeError("No se pudo construir la secuencia de características")
    return result

def temporal_moment(sequence: np.ndarray, order: int = 1) -> np.ndarray:
    """Calcula un momento temporal centrado de una secuencia de vectores."""
    array = np.asarray(sequence, dtype=np.float64)
    if array.ndim != 2 or len(array) == 0:
        raise ValueError("La secuencia debe tener forma tiempo por dimensión")
    if order <= 0:
        raise ValueError("El orden debe ser positivo")

    positions = np.linspace(-1.0, 1.0, len(array)) ** order
    return positions @ array / len(array)

def endpoint_motion_descriptor(sequence: np.ndarray) -> np.ndarray:
    """Resume el cambio neto entre el primer y el último estado."""
    array = np.asarray(sequence, dtype=np.float64)
    if array.ndim != 2 or len(array) < 2:
        raise ValueError("La secuencia debe contener al menos dos instantes")
    return array[-1] - array[0]

def sinusoidal_position_encoding(length: int, dimension: int) -> np.ndarray:
    """Construye una codificación posicional sinusoidal."""
    if length <= 0 or dimension <= 0:
        raise ValueError("La longitud y la dimensión deben ser positivas")

    positions = np.arange(length, dtype=np.float64)[:, None]
    frequencies = np.exp(
        np.arange(0, dimension, 2, dtype=np.float64)
        * (-np.log(10000.0) / dimension)
    )
    encoding = np.zeros((length, dimension), dtype=np.float64)
    encoding[:, 0::2] = np.sin(positions * frequencies)
    encoding[:, 1::2] = np.cos(positions * frequencies[: encoding[:, 1::2].shape[1]])
    return encoding

def attention_pair_counts(
    temporal_tokens: int,
    spatial_tokens: int,
) -> dict[str, int | float]:
    """Compara el número de pares de atención completa y factorizada."""
    if temporal_tokens <= 0 or spatial_tokens <= 0:
        raise ValueError("Las cantidades de tokens deben ser positivas")

    total_tokens = temporal_tokens * spatial_tokens
    full_pairs = total_tokens**2
    divided_pairs = temporal_tokens * (spatial_tokens**2) + spatial_tokens * (
        temporal_tokens**2
    )
    return {
        "temporal_tokens": temporal_tokens,
        "spatial_tokens": spatial_tokens,
        "total_tokens": total_tokens,
        "full_pairs": full_pairs,
        "divided_pairs": divided_pairs,
        "reduction_factor": full_pairs / divided_pairs,
    }

def sample_motion_signal(
    motion_frequency: float,
    duration: float,
    sample_rate: float,
    phase: float = 0.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Muestrea una trayectoria sinusoidal para estudiar aliasing temporal."""
    if motion_frequency <= 0 or duration <= 0 or sample_rate <= 0:
        raise ValueError("Frecuencia, duración y tasa de muestreo deben ser positivas")

    sample_count = int(np.floor(duration * sample_rate))
    times = np.arange(sample_count, dtype=np.float64) / sample_rate
    positions = np.sin(2.0 * np.pi * motion_frequency * times + phase)
    return times, positions

def estimate_dominant_frequency(signal: np.ndarray, sample_rate: float) -> float:
    """Estima la frecuencia dominante de una señal real mediante la FFT."""
    values = np.asarray(signal, dtype=np.float64)
    if values.ndim != 1 or len(values) < 4:
        raise ValueError("La señal debe ser un vector con al menos cuatro muestras")
    if sample_rate <= 0:
        raise ValueError("La tasa de muestreo debe ser positiva")

    centered = values - values.mean()
    spectrum = np.abs(np.fft.rfft(centered))
    frequencies = np.fft.rfftfreq(len(centered), d=1.0 / sample_rate)
    spectrum[0] = 0.0
    return float(frequencies[int(np.argmax(spectrum))])

def mask_tubelets(
    tubelets: np.ndarray,
    mask_ratio: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Oculta una fracción de tubelets y devuelve la máscara aplicada."""
    array = np.asarray(tubelets, dtype=np.float64)
    if array.ndim != 2 or len(array) == 0:
        raise ValueError("Los tubelets deben tener forma tokens por dimensión")
    if not 0.0 <= mask_ratio < 1.0:
        raise ValueError("La proporción de máscara debe estar entre cero y uno")

    rng = np.random.default_rng(seed)
    mask_count = int(round(len(array) * mask_ratio))
    mask = np.zeros(len(array), dtype=bool)
    if mask_count > 0:
        mask[rng.choice(len(array), size=mask_count, replace=False)] = True

    masked = array.copy()
    masked[mask] = 0.0
    return masked, mask

def reconstruct_with_visible_mean(
    masked_tubelets: np.ndarray,
    mask: np.ndarray,
) -> np.ndarray:
    """Reconstruye tokens ocultos mediante el promedio de los tokens visibles."""
    array = np.asarray(masked_tubelets, dtype=np.float64)
    mask_array = np.asarray(mask, dtype=bool)
    if array.ndim != 2 or mask_array.shape != (len(array),):
        raise ValueError("La máscara no coincide con la cantidad de tubelets")
    if mask_array.all():
        raise ValueError("Debe permanecer al menos un tubelet visible")

    reconstructed = array.copy()
    visible_mean = reconstructed[~mask_array].mean(axis=0)
    reconstructed[mask_array] = visible_mean
    return reconstructed

def masked_reconstruction_error(
    original: np.ndarray,
    reconstructed: np.ndarray,
    mask: np.ndarray,
) -> float:
    """Calcula el error cuadrático medio sobre tokens ocultos."""
    original_array = np.asarray(original, dtype=np.float64)
    reconstructed_array = np.asarray(reconstructed, dtype=np.float64)
    mask_array = np.asarray(mask, dtype=bool)
    if original_array.shape != reconstructed_array.shape:
        raise ValueError("Los arreglos original y reconstruido deben coincidir")
    if mask_array.shape != (len(original_array),):
        raise ValueError("La máscara no coincide con los tokens")
    if not mask_array.any():
        return 0.0

    difference = original_array[mask_array] - reconstructed_array[mask_array]
    return float(np.mean(difference**2))

def required_frame_count(duration: float, target_fps: float) -> int:
    """Calcula la cantidad mínima de frames para una duración y tasa objetivo."""
    if duration <= 0 or target_fps <= 0:
        raise ValueError("La duración y la tasa deben ser positivas")
    return int(ceil(duration * target_fps))


SEED = set_seed(211)
RESULTS_DIR = ensure_directory(
    WEEK_DIR / "results" / "cuaderno24_mcc225_avanzado"
)

metadata = ExperimentMetadata(
    course="MCC225",
    week="Semana 12",
    notebook="Cuaderno24-MCC225",
    topic="De imágenes a videos y representación espacio-temporal",
    seed=SEED,
    mode="CPU, datos sintéticos controlados y funciones autocontenidas",
)

save_json(
    metadata_to_dict(metadata),
    RESULTS_DIR / "metadata.json",
)

metadata_to_dict(metadata)


### **Una imagen como señal espacial**

#### **Dominio, muestreo y representación**

Una imagen digital puede modelarse como:

$$
I \in \mathbb{R}^{H \times W \times C},
$$

donde $H$ es el alto, $W$ es el ancho y $C$ es la cantidad de canales.

En una red convolucional 2D, una salida espacial se calcula como:

$$
Y_{h,w,o}
=
\sum_{i=0}^{k_h-1}
\sum_{j=0}^{k_w-1}
\sum_{c=1}^{C}
K^{(2D)}_{i,j,c,o}
I_{h+i,w+j,c}.
$$

Esta operación comparte parámetros sobre el plano espacial. Su sesgo inductivo principal es la localidad espacial.

En un Vision Transformer, la imagen se divide en parches de tamaño $P \times P$. La cantidad de parches es:

$$
N_I
=
\frac{H}{P}
\frac{W}{P}.
$$

Cada parche se aplana y proyecta a dimensión $D$:

$$
x_s
=
\operatorname{vec}(I_s)E_I,
\qquad
E_I
\in
\mathbb{R}^{P^2 C \times D}.
$$

La secuencia inicial puede escribirse como:

$$
Z^{(0)}
=
\left[
x_{\mathrm{cls}},
x_1,
\ldots,
x_{N_I}
\right]
+
E_{\mathrm{pos}}.
$$

El paso a video exige introducir una dimensión temporal y decidir cómo se conectará con el espacio.


### **Experimento 1: tokenización espacial**

#### **Parches, dimensión de token y pérdida de estructura local**

Se toma un frame sintético y se convierte en una secuencia de parches. El experimento permite verificar la relación entre resolución, tamaño de parche, cantidad de tokens y dimensión de cada token.


In [ ]:
video_right = generate_moving_square_video(
    num_frames=16,
    height=64,
    width=64,
    square_size=12,
    direction="derecha",
    noise_level=0.01,
    seed=SEED,
)

image = video_right[0]
patch_size = 8

image_patches = patchify_image(image, patch_size=patch_size)
image_summary = summarize_image_tokens(
    image_shape=image.shape,
    patch_size=patch_size,
)

image_token_table = pd.DataFrame([
    {
        "alto": image.shape[0],
        "ancho": image.shape[1],
        "canales": image.shape[2],
        "tamano_parche": patch_size,
        "tokens_espaciales": image_summary.spatial_tokens,
        "dimension_token": image_summary.token_dimension,
    }
])

image_token_table


In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Frame inicial utilizado como imagen")
plt.axis("off")
plt.show()


In [ ]:
patch_norms = np.linalg.norm(image_patches, axis=1).reshape(
    image.shape[0] // patch_size,
    image.shape[1] // patch_size,
)

plt.figure(figsize=(6, 5))
plt.imshow(patch_norms)
plt.title("Norma de cada token espacial")
plt.xlabel("Índice espacial horizontal")
plt.ylabel("Índice espacial vertical")
plt.colorbar(label="Norma del parche")
plt.show()


### **Un video como señal espacio-temporal**

#### **El video no es solamente una colección de imágenes**

Un video discreto puede representarse como:

$$
V
\in
\mathbb{R}^{T \times H \times W \times C},
$$

donde $T$ es la cantidad de frames.

Una formulación más fundamental parte de una señal continua:

$$
\mathcal{V}(x,y,t),
$$

que se observa únicamente en posiciones y tiempos discretos:

$$
V[n,h,w,c]
=
\mathcal{V}
\left(
h\Delta_x,
w\Delta_y,
n\Delta_t,
c
\right).
$$

La dimensión temporal introduce propiedades que no existen en una imagen aislada:

1. orden,
2. dirección,
3. velocidad,
4. duración,
5. repetición,
6. simultaneidad,
7. cambio de estado,
8. dependencia entre eventos.

Dos videos pueden contener exactamente los mismos frames y tener significados diferentes si el orden cambia.


### **Muestreo temporal y aliasing**

#### **La tasa de frames es una decisión del modelo**

Si la tasa de muestreo es $f_s$, entonces:

$$
\Delta_t
=
\frac{1}{f_s}.
$$

Para una señal temporal ideal con frecuencia máxima $f_{\max}$, una condición clásica para evitar aliasing es:

$$
f_s
>
2 f_{\max}.
$$

En video real, el movimiento no suele ser una sinusoide estacionaria. Sin embargo, la idea sigue siendo relevante: una tasa insuficiente puede cambiar la frecuencia observada, ocultar eventos breves o producir una dirección aparente incorrecta.

Cuando se submuestrea un video, el modelo no recibe una versión más barata de la misma evidencia. Recibe otra observación del fenómeno.


### **Experimento 2: aliasing temporal**

#### **Estimación de frecuencia bajo distintas tasas de muestreo**

Se genera una trayectoria periódica con frecuencia conocida y se estima su frecuencia dominante después del muestreo.


In [ ]:
motion_frequency = 6.0
duration = 2.0
sample_rates = [60.0, 30.0, 15.0, 10.0, 8.0]

aliasing_rows = []
signals_by_rate = {}

for sample_rate in sample_rates:
    times, positions = sample_motion_signal(
        motion_frequency=motion_frequency,
        duration=duration,
        sample_rate=sample_rate,
    )
    estimated_frequency = estimate_dominant_frequency(
        signal=positions,
        sample_rate=sample_rate,
    )
    signals_by_rate[sample_rate] = (times, positions)
    aliasing_rows.append({
        "frecuencia_real_hz": motion_frequency,
        "tasa_muestreo_hz": sample_rate,
        "frecuencia_nyquist_hz": sample_rate / 2.0,
        "frecuencia_estimada_hz": estimated_frequency,
        "error_absoluto_hz": abs(estimated_frequency - motion_frequency),
        "cumple_nyquist": sample_rate > 2.0 * motion_frequency,
    })

aliasing_table = pd.DataFrame(aliasing_rows)
aliasing_table


In [ ]:
plt.figure(figsize=(10, 5))
for sample_rate, (times, positions) in signals_by_rate.items():
    plt.plot(
        times,
        positions,
        marker="o",
        label=f"{sample_rate:.0f} fps",
    )

plt.title("Una misma trayectoria observada con diferentes tasas")
plt.xlabel("Tiempo en segundos")
plt.ylabel("Posición normalizada")
plt.legend()
plt.show()


### **Primera ruta: de convoluciones 2D a convoluciones 3D**

#### **Inflación de kernels espaciales**

Una convolución 3D opera sobre tiempo y espacio:

$$
Y_{t,h,w,o}
=
\sum_{\delta=0}^{k_t-1}
\sum_{i=0}^{k_h-1}
\sum_{j=0}^{k_w-1}
\sum_{c=1}^{C}
K^{(3D)}_{\delta,i,j,c,o}
V_{t+\delta,h+i,w+j,c}.
$$

Una forma de reutilizar un kernel 2D preentrenado consiste en repetirlo sobre el eje temporal:

$$
K^{(3D)}_{\delta,i,j,c,o}
=
\frac{1}{k_t}
K^{(2D)}_{i,j,c,o},
\qquad
\delta
=
0,\ldots,k_t-1.
$$

La división entre $k_t$ preserva la respuesta sobre una entrada estática repetida:

$$
V_t
=
I
\quad
\Longrightarrow
\quad
K^{(3D)} * V
=
K^{(2D)} * I.
$$

Esta propiedad explica la idea de inflación utilizada por I3D. No implica que el kernel ya modele movimiento. Solo proporciona una inicialización compatible con filtros espaciales aprendidos en imágenes.


### **Experimento 3: preservación de la respuesta estática**

#### **Comparación numérica entre convolución 2D y kernel inflado**


In [ ]:
grayscale_image = image.mean(axis=-1)

kernel_2d = np.array([
    [1.0, 0.0, -1.0],
    [2.0, 0.0, -2.0],
    [1.0, 0.0, -1.0],
])

temporal_size = 3
kernel_3d = inflate_2d_kernel(
    kernel_2d=kernel_2d,
    temporal_size=temporal_size,
    preserve_static_response=True,
)

static_video = np.repeat(
    grayscale_image[None, :, :],
    temporal_size,
    axis=0,
)

response_2d = convolve2d_valid(
    image=grayscale_image,
    kernel=kernel_2d,
)

response_3d = convolve3d_valid(
    video=static_video,
    kernel=kernel_3d,
)[0]

static_response_error = float(
    np.max(np.abs(response_2d - response_3d))
)

pd.DataFrame([
    {
        "tamano_temporal": temporal_size,
        "error_maximo": static_response_error,
        "respuesta_preservada": static_response_error < 1e-10,
    }
])


### **Limitación de la inflación**

#### **Preservar filtros espaciales no equivale a modelar movimiento**

Si todos los slices temporales del kernel inflado son iguales, el filtro inicial agrega evidencia sobre tiempo, pero no distingue dirección.

Para detectar una derivada temporal elemental se requiere un kernel con pesos de signos diferentes:

$$
K^{(\Delta t)}
=
[-1,1].
$$

Aplicado a una secuencia de características $h_t$, produce:

$$
\Delta h_t
=
h_t
-
h_{t-1}.
$$

Por ello, el paso de imagen a video necesita dos componentes:

1. transferencia del conocimiento espacial,
2. aprendizaje explícito de variación temporal.


### **Segunda ruta: de parches de imagen a tubelets de video**

#### **Tokenización espacio-temporal**

Para una imagen, un token representa un parche $P \times P$.

Para un video, un tubelet representa un bloque:

$$
\tau
\times
P
\times
P
\times
C,
$$

donde $\tau$ es la extensión temporal.

La cantidad de tokens de video es:

$$
N_V
=
\frac{T}{\tau}
\frac{H}{P}
\frac{W}{P}.
$$

Cada tubelet se proyecta como:

$$
u_{r,s}
=
\operatorname{vec}
\left(
V_{r,s}
\right)
E_V,
$$

con:

$$
E_V
\in
\mathbb{R}^{\tau P^2 C \times D}.
$$

La representación inicial puede factorizar posición espacial y temporal:

$$
z_{r,s}^{(0)}
=
u_{r,s}
+
e_s^{\mathrm{espacio}}
+
e_r^{\mathrm{tiempo}}.
$$

Esta formulación separa tres decisiones:

1. resolución espacial $P$,
2. resolución temporal $\tau$,
3. dimensión latente $D$.

Aumentar $\tau$ reduce tokens, pero puede fusionar eventos breves dentro del mismo tubelet.


### **Experimento 4: crecimiento de tokens**

#### **Comparación entre imagen y video**


In [ ]:
tokenization_rows = []

for current_patch_size in [4, 8, 16]:
    image_info = summarize_image_tokens(
        image_shape=image.shape,
        patch_size=current_patch_size,
    )
    tokenization_rows.append({
        "tipo": "imagen",
        "frames": 1,
        "tamano_temporal": 1,
        "tamano_parche": current_patch_size,
        "tokens_temporales": image_info.temporal_tokens,
        "tokens_espaciales": image_info.spatial_tokens,
        "tokens_totales": image_info.token_count,
        "dimension_token": image_info.token_dimension,
    })

for tubelet_size in [1, 2, 4, 8]:
    video_info = summarize_video_tokens(
        video_shape=video_right.shape,
        tubelet_size=tubelet_size,
        patch_size=8,
    )
    tokenization_rows.append({
        "tipo": "video",
        "frames": video_right.shape[0],
        "tamano_temporal": tubelet_size,
        "tamano_parche": 8,
        "tokens_temporales": video_info.temporal_tokens,
        "tokens_espaciales": video_info.spatial_tokens,
        "tokens_totales": video_info.token_count,
        "dimension_token": video_info.token_dimension,
    })

tokenization_table = pd.DataFrame(tokenization_rows)
tokenization_table


In [ ]:
video_tubelets = tubeletify_video(
    video=video_right,
    tubelet_size=2,
    patch_size=8,
)

pd.DataFrame([
    {
        "cantidad_tubelets": len(video_tubelets),
        "dimension_tubelet": video_tubelets.shape[1],
        "memoria_aproximada_bytes": video_tubelets.nbytes,
    }
])


### **Atención espacio-temporal**

#### **Conectividad y complejidad**

La atención estándar sobre $N$ tokens se define como:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)
V.
$$

Si un video produce $T'$ tokens temporales y $S$ tokens espaciales, entonces:

$$
N
=
T'S.
$$

La atención completa considera:

$$
N^2
=
(T'S)^2
$$

pares de tokens.

Una atención dividida puede aplicar primero atención espacial dentro de cada instante y luego atención temporal para cada posición espacial:

$$
C_{\mathrm{dividida}}
=
T'S^2
+
S(T')^2.
$$

La razón aproximada de reducción es:

$$
R
=
\frac{(T'S)^2}
{T'S^2 + S(T')^2}.
$$

La atención dividida reduce costo, pero introduce un sesgo de factorización. Las interacciones espacio-temporales no ocurren todas en una sola operación.


### **Experimento 5: costo de atención**

#### **Atención completa frente a atención dividida**


In [ ]:
attention_rows = []

for temporal_tokens in [4, 8, 16, 32]:
    for spatial_tokens in [49, 196, 576]:
        attention_rows.append(
            attention_pair_counts(
                temporal_tokens=temporal_tokens,
                spatial_tokens=spatial_tokens,
            )
        )

attention_table = pd.DataFrame(attention_rows)
attention_table


In [ ]:
selected_attention = attention_table[
    attention_table["spatial_tokens"] == 196
].copy()

plt.figure(figsize=(8, 5))
plt.plot(
    selected_attention["temporal_tokens"],
    selected_attention["full_pairs"],
    marker="o",
    label="Atención completa",
)
plt.plot(
    selected_attention["temporal_tokens"],
    selected_attention["divided_pairs"],
    marker="o",
    label="Atención dividida",
)
plt.yscale("log")
plt.title("Crecimiento del número de pares de atención")
plt.xlabel("Tokens temporales")
plt.ylabel("Cantidad de pares en escala logarítmica")
plt.legend()
plt.show()


### **Codificación temporal**

#### **Sin posición, la autoatención no conoce el orden**

Sea $H \in \mathbb{R}^{T \times D}$ una secuencia de features. Una codificación sinusoidal puede expresarse como:

$$
PE(t,2i)
=
\sin
\left(
\frac{t}
{10000^{2i/D}}
\right),
$$

$$
PE(t,2i+1)
=
\cos
\left(
\frac{t}
{10000^{2i/D}}
\right).
$$

La representación de entrada se transforma en:

$$
\widetilde{h}_t
=
h_t
+
PE(t).
$$

Para video puede utilizarse una suma factorizada:

$$
\widetilde{z}_{t,s}
=
z_{t,s}
+
PE_T(t)
+
PE_S(s).
$$

También pueden utilizarse embeddings aprendidos, sesgos relativos o codificaciones basadas en tiempo físico. La elección debe documentarse porque afecta extrapolación a duraciones no observadas.


In [ ]:
position_encoding = sinusoidal_position_encoding(
    length=16,
    dimension=8,
)

plt.figure(figsize=(9, 5))
plt.imshow(position_encoding, aspect="auto")
plt.title("Codificación posicional temporal sinusoidal")
plt.xlabel("Dimensión")
plt.ylabel("Índice temporal")
plt.colorbar(label="Valor")
plt.show()


### **El video no es una bolsa de frames**

#### **Demostración de la invariancia del promedio**

Sea una permutación $\pi$ de los índices temporales. El promedio cumple:

$$
\frac{1}{T}
\sum_{t=1}^{T}
h_t
=
\frac{1}{T}
\sum_{t=1}^{T}
h_{\pi(t)}.
$$

Por lo tanto:

$$
\operatorname{MeanPool}(H)
=
\operatorname{MeanPool}
\left(
P_{\pi}H
\right).
$$

Esta igualdad implica que el promedio no puede distinguir:

1. avance y retroceso,
2. causa y efecto,
3. inicio y final,
4. orden correcto y orden barajado.

Un encoder de imágenes seguido por promedio temporal puede reconocer objetos y escenas, pero no garantiza comprensión temporal.


### **Experimento 6: movimiento con los mismos frames**

#### **Dirección original, dirección invertida y secuencia barajada**

Se construye un video donde un cuadrado se desplaza de izquierda a derecha. La versión invertida contiene exactamente los mismos frames en orden contrario.


In [ ]:
video_left = video_right[::-1].copy()

rng = np.random.default_rng(SEED)
shuffle_indices = rng.permutation(len(video_right))
video_shuffled = video_right[shuffle_indices]

selected_indices = [0, 3, 6, 9, 12, 15]
contact_sheet = np.concatenate(
    [video_right[index] for index in selected_indices],
    axis=1,
)

plt.figure(figsize=(14, 3))
plt.imshow(contact_sheet)
plt.title("Frames seleccionados del movimiento de izquierda a derecha")
plt.axis("off")
plt.show()


In [ ]:
representations = {
    "derecha": extract_frame_features(video_right),
    "izquierda": extract_frame_features(video_left),
    "barajado": extract_frame_features(video_shuffled),
}

representation_rows = []

for name, features in representations.items():
    mean_representation = mean_pool(features)
    temporal_representation = position_aware_pool(features)
    first_moment = temporal_moment(features, order=1)
    motion_descriptor = endpoint_motion_descriptor(features)
    difference_energy = float(
        np.mean(frame_difference(
            {
                "derecha": video_right,
                "izquierda": video_left,
                "barajado": video_shuffled,
            }[name]
        ) ** 2)
    )

    representation_rows.append({
        "secuencia": name,
        "centro_x_promedio": mean_representation[1],
        "momento_temporal_x": first_moment[1],
        "desplazamiento_x": motion_descriptor[1],
        "energia_diferencias": difference_energy,
        "dimension_position_aware": len(temporal_representation),
    })

representation_table = pd.DataFrame(representation_rows)
representation_table


In [ ]:
plt.figure(figsize=(9, 5))

for name, features in representations.items():
    plt.plot(
        features[:, 1],
        marker="o",
        label=name,
    )

plt.title("Trayectoria horizontal estimada por frame")
plt.xlabel("Índice temporal")
plt.ylabel("Centroide horizontal normalizado")
plt.legend()
plt.show()


### **Interpretación del experimento de orden**

#### **Qué conserva cada descriptor**

El centroide promedio es casi idéntico para las tres secuencias porque todas contienen los mismos frames.

El momento temporal de primer orden puede expresarse como:

$$
m_1
=
\frac{1}{T}
\sum_{t=1}^{T}
\widetilde{t}
h_t,
$$

donde:

$$
\widetilde{t}
\in
[-1,1].
$$

El desplazamiento entre extremos es:

$$
d
=
h_T
-
h_1.
$$

Ambos descriptores cambian cuando se invierte la secuencia. Sin embargo, tienen limitaciones:

1. el momento temporal resume toda la trayectoria en un vector,
2. el descriptor de extremos ignora movimientos intermedios,
3. la energía de diferencias detecta cambio, pero puede no distinguir dirección,
4. ninguna de estas líneas base reemplaza una arquitectura temporal aprendida.

Su valor es metodológico: permiten construir pruebas unitarias y líneas base interpretables.


### **Análisis estadístico**

#### **Sensibilidad al orden bajo ruido y múltiples semillas**

Un solo video sintético no permite estimar variabilidad. Se generan videos con ruido bajo diferentes semillas y se calcula la distancia entre la representación original y la invertida.

Para un método $g$, se define:

$$
D_g
=
\left\|
g(V)
-
g(\operatorname{reverse}(V))
\right\|_2.
$$

Si $D_g$ es cercano a cero, el método es insensible a la inversión. Se estiman intervalos bootstrap del 95 por ciento.


In [ ]:
order_rows = []

for seed in range(40):
    current_video = generate_moving_square_video(
        num_frames=16,
        height=64,
        width=64,
        square_size=12,
        direction="derecha",
        noise_level=0.02,
        seed=seed,
    )
    current_features = extract_frame_features(current_video)
    reversed_features = current_features[::-1]

    methods = {
        "promedio": (
            mean_pool(current_features),
            mean_pool(reversed_features),
        ),
        "momento_temporal": (
            temporal_moment(current_features),
            temporal_moment(reversed_features),
        ),
        "extremos": (
            endpoint_motion_descriptor(current_features),
            endpoint_motion_descriptor(reversed_features),
        ),
        "posicion_contenido": (
            position_aware_pool(current_features),
            position_aware_pool(reversed_features),
        ),
    }

    for method_name, (original, reversed_value) in methods.items():
        order_rows.append({
            "seed": seed,
            "metodo": method_name,
            "distancia_orden": float(
                np.linalg.norm(original - reversed_value)
            ),
        })

order_table = pd.DataFrame(order_rows)

bootstrap_rows = []
for method_name, group in order_table.groupby("metodo"):
    mean_value, lower, upper = bootstrap_mean_interval(
        group["distancia_orden"],
        confidence=0.95,
        iterations=3000,
        seed=SEED,
    )
    bootstrap_rows.append({
        "metodo": method_name,
        "media": mean_value,
        "limite_inferior_95": lower,
        "limite_superior_95": upper,
    })

bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table


In [ ]:
plt.figure(figsize=(9, 5))

ordered_bootstrap = bootstrap_table.sort_values("media")
errors = np.vstack([
    ordered_bootstrap["media"] - ordered_bootstrap["limite_inferior_95"],
    ordered_bootstrap["limite_superior_95"] - ordered_bootstrap["media"],
])

plt.errorbar(
    ordered_bootstrap["metodo"],
    ordered_bootstrap["media"],
    yerr=errors,
    fmt="o",
    capsize=5,
)
plt.title("Sensibilidad a la inversión con intervalo bootstrap")
plt.xlabel("Método")
plt.ylabel("Distancia entre secuencia original e invertida")
plt.xticks(rotation=20)
plt.show()


### **Masked video modeling**

#### **Redundancia temporal y reconstrucción de tubelets**

Una estrategia autosupervisada oculta un conjunto $M$ de tokens y entrena un decoder para reconstruirlos:

$$
\mathcal{L}_{\mathrm{MVM}}
=
\frac{1}{|M|}
\sum_{i \in M}
\left\|
x_i
-
\widehat{x}_i
\right\|_2^2.
$$

En video existe alta redundancia temporal. Esto permite utilizar proporciones de máscara superiores a las habituales en imágenes.

Sin embargo, una reconstrucción de bajo error no garantiza comprensión semántica. Un modelo puede explotar continuidad local, fondos estáticos o interpolación de bajo nivel.


### **Experimento 7: reconstrucción básica de tubelets**

#### **Línea base con promedio de tokens visibles**

La línea base no pretende competir con VideoMAE. Su función es mostrar que una métrica de reconstrucción depende de la proporción de máscara y de la redundancia de los datos.


In [ ]:
tubelets = tubeletify_video(
    video=video_right,
    tubelet_size=2,
    patch_size=8,
)

masking_rows = []

for mask_ratio in [0.25, 0.50, 0.75, 0.90]:
    masked_tubelets, mask = mask_tubelets(
        tubelets=tubelets,
        mask_ratio=mask_ratio,
        seed=SEED,
    )
    reconstructed = reconstruct_with_visible_mean(
        masked_tubelets=masked_tubelets,
        mask=mask,
    )
    reconstruction_error = masked_reconstruction_error(
        original=tubelets,
        reconstructed=reconstructed,
        mask=mask,
    )

    masking_rows.append({
        "proporcion_mascara": mask_ratio,
        "tokens_ocultos": int(mask.sum()),
        "tokens_visibles": int((~mask).sum()),
        "error_reconstruccion": reconstruction_error,
    })

masking_table = pd.DataFrame(masking_rows)
masking_table


### **Atención temporal condicionada por consulta**

#### **La representación depende de la pregunta**

Para una consulta $q$ y features temporales $h_t$, una atención simple puede escribirse como:

$$
e_t
=
\frac{
q^\top h_t
}{
\sqrt{D}
},
$$

$$
\alpha_t
=
\frac{
\exp(e_t)
}{
\sum_{j=1}^{T}
\exp(e_j)
},
$$

$$
z_q
=
\sum_{t=1}^{T}
\alpha_t h_t.
$$

La consulta puede seleccionar un instante relevante en lugar de resumir todo el video de manera uniforme.

En tareas reales, $q$ proviene del lenguaje y $h_t$ de un encoder visual. El entrenamiento debe alinear ambos espacios.


In [ ]:
frame_features = extract_frame_features(video_right)

query_right_side = np.array([0.0, 1.0, 0.0, 0.0, 0.0])
query_right_side = query_right_side / np.linalg.norm(query_right_side)

attended_representation, attention_weights = attention_pool(
    sequence=frame_features,
    query=query_right_side,
)

attention_summary = pd.DataFrame([
    {
        "instante_maximo": int(np.argmax(attention_weights)),
        "peso_maximo": float(attention_weights.max()),
        "centro_x_atendido": float(attended_representation[1]),
        "centro_x_promedio": float(frame_features[:, 1].mean()),
        "entropia_atencion": float(
            -np.sum(
                attention_weights
                * np.log(np.maximum(attention_weights, 1e-12))
            )
        ),
    }
])

attention_summary


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(
    attention_weights,
    marker="o",
)
plt.title("Pesos de atención para una consulta sobre el lado derecho")
plt.xlabel("Índice temporal")
plt.ylabel("Peso de atención")
plt.show()


### **De video a video-texto y audio-video**

#### **Alineamiento temporal entre modalidades**

Una vez obtenida una secuencia visual:

$$
H^V
=
[h_1^V,\ldots,h_T^V],
$$

y una secuencia de audio:

$$
H^A
=
[h_1^A,\ldots,h_M^A],
$$

puede construirse una matriz de similitud:

$$
S_{t,m}
=
\frac{
(h_t^V)^\top h_m^A
}{
\|h_t^V\|_2
\|h_m^A\|_2
}.
$$

Si ambas modalidades comparten una base temporal, puede estudiarse una diagonal de correspondencia. Si sus frecuencias son diferentes, deben alinearse timestamps o utilizarse métodos de alineamiento flexible.

Para texto, una consulta $q$ puede utilizarse para producir una distribución temporal:

$$
p(t \mid q,V)
=
\operatorname{softmax}
\left(
s(q,h_t^V)
\right).
$$

El grounding temporal busca un intervalo:

$$
\widehat{I}
=
[\widehat{t}_s,\widehat{t}_e],
$$

y puede evaluarse mediante:

$$
\operatorname{tIoU}
=
\frac{
|\widehat{I} \cap I|
}{
|\widehat{I} \cup I|
}.
$$

La correspondencia semántica y la sincronía no son equivalentes. Un audio puede pertenecer a la misma escena y estar desplazado varios segundos.


### **Taxonomía de arquitecturas**

#### **Decisiones de diseño al pasar de imagen a video**

| Familia | Conversión desde imagen | Sesgo temporal | Ventaja | Riesgo |
|---|---|---|---|---|
| Encoder 2D por frame | Aplica el mismo encoder a cada frame | Depende del agregador | Reutiliza modelos de imagen | Puede ignorar el orden |
| CNN 3D | Infla o aprende kernels espacio-temporales | Localidad temporal | Modela movimiento local | Costo alto y horizonte limitado |
| Dos rutas temporales | Usa tasas de muestreo diferentes | Movimiento lento y rápido | Separa semántica y dinámica | Fusión y calibración complejas |
| Transformer completo | Tokeniza todo el volumen | Conectividad global | Interacciones generales | Costo cuadrático |
| Transformer factorizado | Separa espacio y tiempo | Factorización explícita | Menor costo | Interacción indirecta |
| Atención local | Ventanas espacio-temporales | Localidad jerárquica | Escalabilidad | Puede perder relaciones largas |
| Masked video model | Reconstruye tokens ocultos | Redundancia espacio-temporal | Preentrenamiento autosupervisado | Puede aprender atajos locales |
| Video foundation model | Combina objetivos y datos multimodales | Depende del diseño | Transferencia a varias tareas | Evaluación y trazabilidad difíciles |


### **Amenazas a la validez**

#### **Interpretación limitada por la evidencia**

1. Los videos sintéticos aíslan dirección, orden y muestreo, pero no representan oclusión, cámara móvil ni iluminación real.
2. El centroide es interpretable, pero presupone que el objeto puede separarse del fondo.
3. El promedio visible utilizado en masked modeling es una línea base, no un decoder aprendido.
4. El conteo de pares de atención aproxima costo estructural, pero no sustituye mediciones de memoria y tiempo.
5. La inflación de kernels preserva respuesta estática, pero no garantiza transferencia óptima a movimiento.
6. La sensibilidad a inversión es necesaria para algunas tareas, pero no para todas.
7. Un modelo puede resolver benchmarks de video usando objetos, escenas, audio o subtítulos sin modelar orden.
8. Una respuesta correcta no demuestra grounding temporal.


### **Conclusiones**

#### **Respuesta a la pregunta central**

El paso de imágenes a videos requiere más que añadir frames.

1. **La entrada cambia:** de un tensor espacial a un tensor espacio-temporal.
2. **El muestreo cambia:** la tasa de frames determina qué fenómenos son observables.
3. **La arquitectura cambia:** se introducen kernels 3D, tubelets, posición temporal o mecanismos de atención temporal.
4. **El costo cambia:** la secuencia crece con frames y parches.
5. **La evaluación cambia:** deben medirse orden, duración, dirección, sincronía y localización.
6. **La metodología cambia:** una línea base de imagen debe compararse con perturbaciones temporales controladas.

Un modelo de video debe demostrar que su salida depende de relaciones entre instantes. Sin esa evidencia, solo puede afirmarse que procesa múltiples imágenes.


#### **Preguntas a desarrollar**

1. ¿Por qué inflar un kernel 2D preserva la respuesta estática?
2. ¿Qué propiedad temporal no puede representar el pooling promedio?
3. ¿Qué se pierde al aumentar la extensión temporal de un tubelet?
4. ¿Cuándo conviene atención completa y cuándo atención dividida?
5. ¿Cómo se demostraría que un modelo utiliza orden y no solo objetos?
6. ¿Por qué una energía de diferencias puede detectar movimiento sin identificar dirección?
7. ¿Qué relación existe entre tasa de muestreo y eventos breves?
8. ¿Cómo se separaría grounding temporal de razonamiento temporal?
9. ¿Qué atajos permitirían resolver una pregunta de video usando un frame?
10. ¿Qué experimento distinguiría correspondencia audiovisual de sincronía?
11. ¿Por qué una pérdida de reconstrucción baja no demuestra comprensión?
12. ¿Cómo se adaptaría este protocolo a videos largos?.


In [ ]:
# Tus respuestas

### **Referencias principales**


1. Dosovitskiy et al. **An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale**. [arXiv](https://arxiv.org/abs/2010.11929)
2. Carreira y Zisserman. **Quo Vadis, Action Recognition? A New Model and the Kinetics Dataset**. [arXiv](https://arxiv.org/abs/1705.07750)
3. Feichtenhofer et al. **SlowFast Networks for Video Recognition**. [arXiv](https://arxiv.org/abs/1812.03982)
4. Bertasius et al. **Is Space-Time Attention All You Need for Video Understanding?** [arXiv](https://arxiv.org/abs/2102.05095)
5. Arnab et al. **ViViT: A Video Vision Transformer**. [arXiv](https://arxiv.org/abs/2103.15691)
6. Liu et al. **Video Swin Transformer**. [arXiv](https://arxiv.org/abs/2106.13230)
7. Tong et al. **VideoMAE: Masked Autoencoders are Data-Efficient Learners for Self-Supervised Video Pre-Training**. [arXiv](https://arxiv.org/abs/2203.12602)
8. Wang et al. **InternVideo2: Scaling Foundation Models for Multimodal Video Understanding**. [arXiv](https://arxiv.org/abs/2403.15377)


### **Exportación de resultados**

#### **Evidencia reproducible**


In [ ]:
save_csv(
    image_token_table.to_dict(orient="records"),
    RESULTS_DIR / "image_tokens.csv",
)
save_csv(
    aliasing_table.to_dict(orient="records"),
    RESULTS_DIR / "temporal_aliasing.csv",
)
save_csv(
    tokenization_table.to_dict(orient="records"),
    RESULTS_DIR / "tokenization_comparison.csv",
)
save_csv(
    attention_table.to_dict(orient="records"),
    RESULTS_DIR / "attention_costs.csv",
)
save_csv(
    representation_table.to_dict(orient="records"),
    RESULTS_DIR / "order_representations.csv",
)
save_csv(
    order_table.to_dict(orient="records"),
    RESULTS_DIR / "order_trials.csv",
)
save_csv(
    bootstrap_table.to_dict(orient="records"),
    RESULTS_DIR / "order_bootstrap.csv",
)
save_csv(
    masking_table.to_dict(orient="records"),
    RESULTS_DIR / "masked_video_baseline.csv",
)

summary = {
    "error_respuesta_estatica": static_response_error,
    "metodo_mas_sensible_orden": (
        bootstrap_table
        .sort_values("media", ascending=False)
        .iloc[0]["metodo"]
    ),
    "frecuencia_real_hz": motion_frequency,
    "tasa_minima_evaluada_hz": min(sample_rates),
    "tokens_video_tubelet_2": int(len(video_tubelets)),
    "directorio_resultados": str(RESULTS_DIR),
}

save_json(
    summary,
    RESULTS_DIR / "summary.json",
)

summary
